# MBPP OOD + TACO Robustness Evaluation

This notebook clones the repo into Colab, installs the evaluation package, optionally discovers your Hugging Face model repos, validates which repos are real HF-loadable models, and runs the OOD + TACO robustness workflow against the valid subset.

In [ ]:
GIT_REPO_URL = "https://github.com/samuelleecong/CS4248_ez_A.git"
GIT_BRANCH = "ood_perturb"
REPO_DIR = "/content/CS4248_ez_A"

GITHUB_TOKEN = ""  # or use Colab Secret named GITHUB_TOKEN

import os
from urllib.parse import urlparse

try:
    from google.colab import userdata  # type: ignore
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN") or ""
except Exception:
    pass

if not GITHUB_TOKEN:
    raise ValueError("Set GITHUB_TOKEN directly or in Colab Secrets.")

%cd /content

parsed = urlparse(GIT_REPO_URL)
auth_repo_url = f"https://oauth2:{GITHUB_TOKEN}@{parsed.netloc}{parsed.path}"

!rm -rf "$REPO_DIR"
!git clone --branch "$GIT_BRANCH" "$auth_repo_url" "$REPO_DIR"

%cd $REPO_DIR
!python -m pip install -q -U pip
!python -m pip install -q textattack
!python -m pip install -q -e ./mbpp_kd_suite

In [ ]:
# Optional Hugging Face discovery/auth setup.
# Set HF_OWNER to your username or org if you want the notebook to list your model repos.
# For private repos, provide HF_TOKEN directly or via Colab Secrets named HF_TOKEN.

HF_OWNER = "cs4248-nlp"
HF_TOKEN = ""
AUTO_DISCOVER_MODELS = True
MODEL_NAME_FILTER = ""

import os

try:
    from google.colab import userdata  # type: ignore
    if not HF_TOKEN:
        HF_TOKEN = userdata.get("HF_TOKEN") or ""
except Exception:
    pass

if HF_TOKEN:
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
from huggingface_hub import HfApi

discovered_model_ids = []
if AUTO_DISCOVER_MODELS:
    if not HF_OWNER:
        raise ValueError("Set HF_OWNER when AUTO_DISCOVER_MODELS=True.")
    api = HfApi(token=HF_TOKEN or None)
    discovered = list(api.list_models(author=HF_OWNER, full=True))
    discovered_model_ids = [model.id for model in discovered if model.id]
    if MODEL_NAME_FILTER:
        discovered_model_ids = [model_id for model_id in discovered_model_ids if MODEL_NAME_FILTER.lower() in model_id.lower()]
    discovered_model_ids = sorted(discovered_model_ids)
    print("Discovered models:")
    for model_id in discovered_model_ids:
        print(" -", model_id)
else:
    print("AUTO_DISCOVER_MODELS=False; skipping Hugging Face repo discovery.")

In [ ]:
MODEL_IDS = discovered_model_ids or [
    "sentence-transformers/all-MiniLM-L6-v2",
    # "cs4248-nlp/ft-student-all-minilm-l6-v2-taco-20260326-110507",
]

# Optional: point these at repo-local files if you committed custom data.
MBPP_DATASET_PATH = None
TACO_DATASET_NAME = "BEE-spoke-data/TACO-hf"
TACO_DATASET_PATH = None
SPLIT = "test"
SPLIT_SEED = 42
PERTURBATION_TIER = "all"
OUTPUT_DIR = "./colab_runs/ood_robustness"

assert MODEL_IDS, "Add at least one Hugging Face model ID to MODEL_IDS."

In [ ]:
# Validate discovered/manual models and keep only HF-loadable repos.
from transformers import AutoConfig, AutoTokenizer, AutoModel

valid_model_ids = []
invalid_model_rows = []

for model_id in MODEL_IDS:
    try:
        AutoConfig.from_pretrained(model_id, token=HF_TOKEN or None)
        AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN or None)
        AutoModel.from_pretrained(model_id, token=HF_TOKEN or None)
        valid_model_ids.append(model_id)
        print("VALID:", model_id)
    except Exception as exc:
        invalid_model_rows.append({"model_id": model_id, "error": str(exc)})
        print("SKIP :", model_id)
        print("ERROR:", str(exc)[:250], "\n")

MODEL_IDS = valid_model_ids
print(f"Kept {len(MODEL_IDS)} valid models, skipped {len(invalid_model_rows)}.")
assert MODEL_IDS, "No compatible HF models found after validation."

In [ ]:
import pandas as pd

invalid_df = pd.DataFrame(invalid_model_rows)
invalid_df

In [ ]:
import shlex

model_args = " ".join(f"--model {shlex.quote(model_id)}" for model_id in MODEL_IDS)
mbpp_arg = "" if MBPP_DATASET_PATH is None else f" --mbpp-dataset-path {shlex.quote(MBPP_DATASET_PATH)}"
taco_path_arg = "" if TACO_DATASET_PATH is None else f" --taco-dataset-path {shlex.quote(TACO_DATASET_PATH)}"

cmd = (
    f"PYTHONPATH=/content/CS4248_ez_A/mbpp_kd_suite:/content/CS4248_ez_A/mbpp_kd_suite/src "
    f"python -m eval.ood_robustness {model_args}"
    f" --task all"
    f" --taco-dataset-name {shlex.quote(TACO_DATASET_NAME)}"
    f" --split {shlex.quote(SPLIT)}"
    f" --split-seed {SPLIT_SEED}"
    f" --perturbation-tier {shlex.quote(PERTURBATION_TIER)}"
    f" --output-dir {shlex.quote(OUTPUT_DIR)}"
    f"{mbpp_arg}{taco_path_arg}"
)
print(cmd)
!cd /content/CS4248_ez_A && {cmd}

In [ ]:
from pathlib import Path
import pandas as pd

run_root = Path(OUTPUT_DIR)
latest_run = sorted(run_root.iterdir())[-1]
metrics = pd.read_csv(latest_run / "metrics.csv")
metrics

In [ ]:
import matplotlib.pyplot as plt

taco = metrics[metrics["task"] == "taco_robustness"].copy()
taco_clean = taco[taco["perturbation_tier"] == "clean"].copy()
taco_noisy = taco[taco["perturbation_tier"] != "clean"].copy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(taco_clean["model_name"], taco_clean["mrr"])
axes[0].set_title("TACO Clean MRR")
axes[0].tick_params(axis="x", rotation=35)

for model_name, frame in taco_noisy.groupby("model_name"):
    axes[1].plot(frame["perturbation_tier"], frame["delta_mrr_vs_clean"], marker="o", label=model_name)
axes[1].set_title("TACO Robustness Drop (MRR)")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend()

mbpp = metrics[metrics["task"] == "mbpp_ood"].copy()
compare = taco_clean[["model_name", "mrr"]].rename(columns={"mrr": "taco_clean_mrr"}).merge(
    mbpp[["model_name", "mrr"]].rename(columns={"mrr": "mbpp_ood_mrr"}),
    on="model_name",
    how="outer",
)
compare = compare.set_index("model_name")
compare.plot(kind="bar", ax=axes[2])
axes[2].set_title("MBPP OOD vs TACO Clean")
axes[2].tick_params(axis="x", rotation=35)

plt.tight_layout()
plt.show()